# 04 - Train/test split

Objetivo: construir la base modelable aplicando las decisiones del EDA y separar train/test antes de cualquier encoding.

Aqui se filtra `Monthly_Charge < 0`, se imputan nulos estructurales, se excluyen columnas de leakage y se reserva `Joined` para scoring.

In [1]:
import os

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "../data/Customer_Data.csv"
PROCESSED_DIR = "../data/processed"

TARGET_COL = "Customer_Status"
CUSTOMER_ID_COL = "Customer_ID"
CHURN_CATEGORY_COL = "Churn_Category"
CHURN_REASON_COL = "Churn_Reason"
MONTHLY_CHARGE_COL = "Monthly_Charge"

INTERNET_DEPENDENT_COLS = [
    "Internet_Type",
    "Online_Security",
    "Online_Backup",
    "Device_Protection_Plan",
    "Premium_Support",
    "Streaming_TV",
    "Streaming_Movies",
    "Streaming_Music",
    "Unlimited_Data",
]

# Separadas aparte (no en LEAKAGE_COLS) porque no son leakage en sentido estricto -- son
# predictoras validas para Stayed/Churned -- pero estan fuera de rango para Joined, que aun
# no ha tenido tiempo de acumular estos montos (diagnostico con datos en
# 03_statistical_analysis.ipynb).
CUMULATIVE_SINCE_SIGNUP_COLS = [
    "Total_Charges",
    "Total_Revenue",
    "Total_Refunds",
    "Total_Extra_Data_Charges",
    "Total_Long_Distance_Charges",
]

# Customer_ID no aporta senal como feature, pero se conserva en df_joined para poder
# identificar a cada cliente al puntuarlo en 09_business_insights.ipynb -- se descarta como
# feature recien en 05_feature_engineering.ipynb. Churn_Category/Churn_Reason solo existen
# para clientes que ya churnearon, son consecuencia del churn, no un predictor -- incluirlas
# seria leakage real, y no aplican a Joined (no tiene esas columnas pobladas).
LEAKAGE_COLS_MODEL = [CUSTOMER_ID_COL, CHURN_CATEGORY_COL, CHURN_REASON_COL]
LEAKAGE_COLS_JOINED = [CHURN_CATEGORY_COL, CHURN_REASON_COL]

df = pd.read_csv(DATA_PATH)
df_model = df[df[TARGET_COL].isin(["Stayed", "Churned"])].copy()
df_joined = df[df[TARGET_COL] == "Joined"].copy()

print(f"df_model  (Stayed/Churned, para entrenar/evaluar): {df_model.shape[0]} filas")
print(f"df_joined (Joined, se puntuan al final en 09):     {df_joined.shape[0]} filas")

df_model  (Stayed/Churned, para entrenar/evaluar): 6007 filas
df_joined (Joined, se puntuan al final en 09):     411 filas


## Filtrar `Monthly_Charge` negativo e imputar nulos estructurales

In [2]:
negative_mask = df_model[MONTHLY_CHARGE_COL] < 0
negative_mask_joined = df_joined[MONTHLY_CHARGE_COL] < 0
df_model = df_model.loc[~negative_mask].copy()
df_joined = df_joined.loc[~negative_mask_joined].copy()
print(f"Filas excluidas por Monthly_Charge negativo -- df_model: {negative_mask.sum()}, df_joined: {negative_mask_joined.sum()}")

for frame in (df_model, df_joined):
    for col in INTERNET_DEPENDENT_COLS:
        frame[col] = frame[col].fillna("No Internet Service")
    frame["Multiple_Lines"] = frame["Multiple_Lines"].fillna("No Phone Service")
    frame["Value_Deal"] = frame["Value_Deal"].fillna("No Deal")

Filas excluidas por Monthly_Charge negativo -- df_model: 101, df_joined: 6


Estas reglas no aprenden parametros del dataset: son decisiones fijas de limpieza. Por eso pueden aplicarse antes del split sin generar leakage.

El encoding categorico se deja para `05_feature_engineering.ipynb`, ajustado solo con train.

## Excluir columnas de leakage y las acumuladas desde el alta

In [3]:
print(f"Columnas de leakage excluidas de df_model: {LEAKAGE_COLS_MODEL}")
print(f"Columnas de leakage excluidas de df_joined: {LEAKAGE_COLS_JOINED}")
print(f"Columnas acumuladas desde el alta excluidas: {CUMULATIVE_SINCE_SIGNUP_COLS}")

df_model = df_model.drop(columns=LEAKAGE_COLS_MODEL + CUMULATIVE_SINCE_SIGNUP_COLS)
df_joined = df_joined.drop(columns=LEAKAGE_COLS_JOINED + CUMULATIVE_SINCE_SIGNUP_COLS)

Columnas de leakage excluidas de df_model: ['Customer_ID', 'Churn_Category', 'Churn_Reason']
Columnas de leakage excluidas de df_joined: ['Churn_Category', 'Churn_Reason']
Columnas acumuladas desde el alta excluidas: ['Total_Charges', 'Total_Revenue', 'Total_Refunds', 'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges']


## Train/test split

`stratify=y` mantiene la proporcion de churn en train y test. `random_state=42` hace reproducible la particion.

El split ocurre antes del encoding para que test no influya en las columnas aprendidas por el modelo.

In [4]:
X = df_model.drop(columns=[TARGET_COL])
y = (df_model[TARGET_COL] == "Churned").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} filas (churn rate {y_train.mean():.3f})")
print(f"Prueba:        {X_test.shape[0]} filas (churn rate {y_test.mean():.3f})")

Entrenamiento: 4724 filas (churn rate 0.289)
Prueba:        1182 filas (churn rate 0.288)


## Guardar los conjuntos sin codificar

In [5]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_raw = X_train.copy()
train_raw["churn_flag"] = y_train
train_raw.to_parquet(f"{PROCESSED_DIR}/train_raw.parquet", index=False)

test_raw = X_test.copy()
test_raw["churn_flag"] = y_test
test_raw.to_parquet(f"{PROCESSED_DIR}/test_raw.parquet", index=False)

joined_raw = df_joined.drop(columns=[TARGET_COL])
joined_raw.to_parquet(f"{PROCESSED_DIR}/joined_raw.parquet", index=False)

print("Guardado: train_raw.parquet, test_raw.parquet, joined_raw.parquet")

Guardado: train_raw.parquet, test_raw.parquet, joined_raw.parquet
